In [ ]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.4/30.4 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 24.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from google.colab import drive
from unsloth import FastLanguageModel

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/assetmind"
df = pd.read_parquet(f"{BASE}/phmsa_cause.parquet")

model, tok = FastLanguageModel.from_pretrained(
    "unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit", max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)
tok.padding_side = "left"   # needed for batched generation

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Mounted at /content/drive
==((====))==  Unsloth 2026.10.6: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:
CAUSES = {
    "A": ("CORROSION FAILURE", "metal loss from internal or external corrosion, e.g. pitting, wall thinning"),
    "B": ("EQUIPMENT FAILURE", "malfunction or failure of equipment: pumps, valves, seals, gaskets, threaded or "
                               "non-threaded connections, control or relief equipment"),
    "C": ("EXCAVATION DAMAGE", "damage from digging or excavation activity by the operator or a third party"),
    "D": ("INCORRECT OPERATION", "human error: wrong valve position, overfilling, not following procedures, "
                                 "improper installation or operation"),
    "E": ("MATERIAL FAILURE OF PIPE OR WELD", "defects or cracking in the pipe body, seam or weld, from "
                                              "manufacturing, construction or environmental cracking"),
    "F": ("NATURAL FORCE DAMAGE", "earth movement, heavy rain or flooding, lightning, freezing temperatures, high winds"),
    "G": ("OTHER OUTSIDE FORCE DAMAGE", "external damage not from excavation: vehicles, fire or explosion, "
                                        "boats, intentional damage, previous mechanical damage"),
}
cause_list = "\n".join(f"{k}. {name}: {desc}" for k, (name, desc) in CAUSES.items())

def cause_prompt(text):
    return [{"role": "system", "content": "You are a pipeline integrity engineer classifying incident reports."},
            {"role": "user", "content":
                f"Classify the cause of this pipeline incident into exactly one category.\n\n{cause_list}\n\n"
                "If the narrative contains a supplemental or final update, use the cause stated there.\n\n"
                f"Narrative:\n{text}\n\nAnswer with a single letter (A-G) only."}]

def pending_prompt(text):
    return [{"role": "system", "content": "You are a pipeline integrity engineer reviewing incident reports."},
            {"role": "user", "content":
                "Has the cause of this incident been determined?\n"
                "Answer DETERMINED if the narrative states a cause, including in a later supplemental or final update.\n"
                "Answer UNDETERMINED only if the cause is still under investigation and no later conclusion is given.\n\n"
                f"Narrative:\n{text}\n\nAnswer with one word: DETERMINED or UNDETERMINED."}]

In [ ]:
import torch

def generate(message_lists, max_new_tokens=4, batch_size=8):
    outputs = []
    for i in range(0, len(message_lists), batch_size):
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
                   for m in message_lists[i:i + batch_size]]
        enc = tok(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
        outputs += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        if (i // batch_size) % 20 == 0:
            print(f"{min(i + batch_size, len(message_lists))}/{len(message_lists)}")
    return [o.strip() for o in outputs]

def to_cause(answer):
    letter = next((ch for ch in answer.upper() if ch in CAUSES), None)
    return CAUSES[letter][0] if letter else "INVALID"

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

val = df[(df["split"] == "val") & df["main_track"]]
val_pred = [to_cause(a) for a in generate([cause_prompt(t) for t in val["text"]])]
print("Invalid answers:", sum(p == "INVALID" for p in val_pred))
print(f"Validation macro-F1 {f1_score(val['label'], val_pred, average='macro'):.3f}, "
      f"accuracy {accuracy_score(val['label'], val_pred):.3f}")


Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


8/590


Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

168/590


Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

328/590


Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

488/590


Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=4) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Invalid answers: 0
Validation macro-F1 0.594, accuracy 0.698


In [ ]:
from sklearn.metrics import classification_report
print(classification_report(val["label"], val_pred, zero_division=0))
pd.crosstab(val["label"].values, pd.Series(val_pred, name="pred"), normalize="index").round(2)

                                  precision    recall  f1-score   support

               CORROSION FAILURE       0.91      0.83      0.87       153
               EQUIPMENT FAILURE       0.95      0.61      0.74       265
               EXCAVATION DAMAGE       0.34      1.00      0.51        14
             INCORRECT OPERATION       0.48      0.82      0.61        82
MATERIAL FAILURE OF PIPE OR WELD       0.47      0.63      0.54        41
            NATURAL FORCE DAMAGE       0.88      0.37      0.52        19
      OTHER OUTSIDE FORCE DAMAGE       0.26      0.62      0.37        16

                        accuracy                           0.70       590
                       macro avg       0.61      0.70      0.59       590
                    weighted avg       0.81      0.70      0.72       590



pred,CORROSION FAILURE,EQUIPMENT FAILURE,EXCAVATION DAMAGE,INCORRECT OPERATION,MATERIAL FAILURE OF PIPE OR WELD,NATURAL FORCE DAMAGE,OTHER OUTSIDE FORCE DAMAGE
row_0,,,,,,,
CORROSION FAILURE,0.83,0.01,0.08,0.05,0.01,0.00,0.03
EQUIPMENT FAILURE,0.03,0.61,0.01,0.22,0.09,0.00,0.04
EXCAVATION DAMAGE,0.00,0.00,1.00,0.00,0.00,0.00,0.00
INCORRECT OPERATION,0.00,0.07,0.02,0.82,0.01,0.00,0.07
MATERIAL FAILURE OF PIPE OR WELD,0.10,0.00,0.20,0.02,0.63,0.00,0.05
NATURAL FORCE DAMAGE,0.00,0.11,0.00,0.16,0.11,0.37,0.26
OTHER OUTSIDE FORCE DAMAGE,0.06,0.00,0.12,0.12,0.00,0.06,0.62


In [ ]:
CAUSES = {
    "A": ("CORROSION FAILURE", "metal loss from internal or external corrosion, e.g. pitting, wall thinning"),
    "B": ("EQUIPMENT FAILURE", "failure of equipment: pumps, valves, seals, gaskets, threaded or non-threaded "
                               "connections, control or relief equipment. Use this even if people were working on "
                               "the equipment, unless their own mistake caused the release"),
    "C": ("EXCAVATION DAMAGE", "the pipe was DAMAGED by digging or excavation. Excavation done afterwards to find "
                               "or repair a leak does NOT make this excavation damage"),
    "D": ("INCORRECT OPERATION", "a person's action or omission directly caused the release: wrong valve position, "
                                 "overfilling, not following procedures, improper installation"),
    "E": ("MATERIAL FAILURE OF PIPE OR WELD", "defects or cracking in the pipe body, seam or weld, from "
                                              "manufacturing, construction or environmental cracking"),
    "F": ("NATURAL FORCE DAMAGE", "weather or natural events: freezing temperatures, heavy rain or flooding, "
                                  "lightning, high winds, earth movement. Weather counts here, not as outside force"),
    "G": ("OTHER OUTSIDE FORCE DAMAGE", "external damage by people or objects, not excavation and not weather: "
                                        "vehicles, fire or explosion, boats, intentional damage, previous mechanical damage"),
}
cause_list = "\n".join(f"{k}. {name}: {desc}" for k, (name, desc) in CAUSES.items())

In [ ]:
from transformers import logging
logging.set_verbosity_error()

val_pred_v2 = [to_cause(a) for a in generate([cause_prompt(t) for t in val["text"]])]
print("Invalid answers:", sum(p == "INVALID" for p in val_pred_v2))
print(f"Prompt v2 validation macro-F1 {f1_score(val['label'], val_pred_v2, average='macro'):.3f}, "
      f"accuracy {accuracy_score(val['label'], val_pred_v2):.3f}")
print(classification_report(val["label"], val_pred_v2, zero_division=0))

8/590
168/590
328/590
488/590
Invalid answers: 0
Prompt v2 validation macro-F1 0.550, accuracy 0.639
                                  precision    recall  f1-score   support

               CORROSION FAILURE       0.90      0.81      0.85       153
               EQUIPMENT FAILURE       0.97      0.48      0.64       265
               EXCAVATION DAMAGE       0.26      0.43      0.32        14
             INCORRECT OPERATION       0.49      0.84      0.62        82
MATERIAL FAILURE OF PIPE OR WELD       0.44      0.66      0.52        41
            NATURAL FORCE DAMAGE       0.69      0.58      0.63        19
      OTHER OUTSIDE FORCE DAMAGE       0.15      0.75      0.26        16

                        accuracy                           0.64       590
                       macro avg       0.56      0.65      0.55       590
                    weighted avg       0.80      0.64      0.67       590



In [ ]:
CAUSES = {
    "A": ("CORROSION FAILURE", "metal loss from internal or external corrosion, e.g. pitting, wall thinning"),
    "B": ("EQUIPMENT FAILURE", "malfunction or failure of equipment: pumps, valves, seals, gaskets, threaded or "
                               "non-threaded connections, control or relief equipment"),
    "C": ("EXCAVATION DAMAGE", "damage from digging or excavation activity by the operator or a third party"),
    "D": ("INCORRECT OPERATION", "human error: wrong valve position, overfilling, not following procedures, "
                                 "improper installation or operation"),
    "E": ("MATERIAL FAILURE OF PIPE OR WELD", "defects or cracking in the pipe body, seam or weld, from "
                                              "manufacturing, construction or environmental cracking"),
    "F": ("NATURAL FORCE DAMAGE", "earth movement, heavy rain or flooding, lightning, freezing temperatures, high winds"),
    "G": ("OTHER OUTSIDE FORCE DAMAGE", "external damage not from excavation: vehicles, fire or explosion, "
                                        "boats, intentional damage, previous mechanical damage"),
}
cause_list = "\n".join(f"{k}. {name}: {desc}" for k, (name, desc) in CAUSES.items())

In [ ]:
test_main = df[(df["split"] == "test") & df["main_track"]]
test_pred = [to_cause(a) for a in generate([cause_prompt(t) for t in test_main["text"]])]
print("Invalid answers:", sum(p == "INVALID" for p in test_pred))
print(f"TEST macro-F1 {f1_score(test_main['label'], test_pred, average='macro'):.3f}, "
      f"accuracy {accuracy_score(test_main['label'], test_pred):.3f}")
print(classification_report(test_main["label"], test_pred, zero_division=0))

8/742
168/742
328/742
488/742
648/742
Invalid answers: 0
TEST macro-F1 0.632, accuracy 0.705
                                  precision    recall  f1-score   support

               CORROSION FAILURE       0.95      0.84      0.89       204
               EQUIPMENT FAILURE       0.95      0.59      0.73       353
               EXCAVATION DAMAGE       0.46      0.95      0.62        19
             INCORRECT OPERATION       0.37      0.83      0.51        69
MATERIAL FAILURE OF PIPE OR WELD       0.46      0.79      0.58        48
            NATURAL FORCE DAMAGE       0.83      0.54      0.65        28
      OTHER OUTSIDE FORCE DAMAGE       0.31      0.71      0.43        21

                        accuracy                           0.70       742
                       macro avg       0.62      0.75      0.63       742
                    weighted avg       0.83      0.70      0.73       742



In [ ]:
from sklearn.metrics import precision_score, recall_score

test_all = df[df["split"] == "test"]
answers = generate([pending_prompt(t) for t in test_all["text"]], max_new_tokens=5)
pend_pred = ["UNDETERMINED" in a.upper() for a in answers]
print("Unclear answers:", sum("DETERMINED" not in a.upper() for a in answers))
print(f"Pending: precision {precision_score(test_all['pending'], pend_pred, zero_division=0):.3f}, "
      f"recall {recall_score(test_all['pending'], pend_pred, zero_division=0):.3f}, "
      f"F1 {f1_score(test_all['pending'], pend_pred, zero_division=0):.3f}")

8/778
168/778
328/778
488/778
648/778
Unclear answers: 0
Pending: precision 0.344, recall 0.939, F1 0.504


In [ ]:
pd.DataFrame({"label": test_main["label"].values, "pred": test_pred}, index=test_main.index).to_parquet(
    f"{BASE}/phmsa_main_zeroshot_test.parquet")
pd.DataFrame({"pending": test_all["pending"].values, "pred": pend_pred, "answer": answers},
             index=test_all.index).to_parquet(f"{BASE}/phmsa_pending_zeroshot_test.parquet")